# SEDD Pipeline Notebook

This notebook runs the full SEDD workflow from Python cells: data preparation, compact from-scratch training, official `louaaron/sedd-small` fine-tuning, RL post-training, evaluation, and inference.

The compact from-scratch model is pretrained on a bounded TinyStories subset. The specialized official-model task is multiple-choice science QA: ARC-Easy is used for supervised fine-tuning and ARC-Challenge is used for RL with exact answer-key reward. The cell outputs are the demonstration surface.

## 0. Overview

- The compact model is trained from scratch on TinyStories to show the absorbing discrete diffusion objective end to end on a real non-SEDD dataset.
- Official SEDD-small is adapted on ARC-Easy with response-only score entropy SFT.
- RL uses ARC-Challenge prompts and answer keys, treating denoising choices as policy actions.
- Evaluation reports validation score entropy and exact multiple-choice reward/accuracy from generated answers.

In [ ]:
from __future__ import annotations

import json
import os
import random
import time
from pathlib import Path
from typing import Any

import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from sedd_mini.backend import GenerationParams
from sedd_mini.checkpoint import load_checkpoint, save_checkpoint
from sedd_mini.config import load_config, save_config
from sedd_mini.data import (
    TOY_SFT_RECORDS,
    TokenDataset,
    build_pretrain_dataset as build_mini_pretrain_dataset,
    build_sft_dataset as build_mini_sft_dataset,
    collate_batch,
    save_dataset as save_mini_dataset,
    split_dataset as split_mini_dataset,
)
from sedd_mini.diffusion import loglinear_noise, score_entropy_loss
from sedd_mini.mcqa_data import (
    MCQARecord,
    as_sft_records,
    exact_choice_reward,
    extract_choice,
    load_arc_records,
    load_mcqa_records,
    save_mcqa_records,
)
from sedd_mini.model import build_model
from sedd_mini.official_backend import OfficialSEDDBackend, load_official_components, save_official_checkpoint
from sedd_mini.official_finetune import apply_lora, evaluate as evaluate_official_loss
from sedd_mini.official_finetune import official_score_entropy_loss, save_merged_lora_checkpoint, total_parameter_count, trainable_parameter_count
from sedd_mini.official_posttrain_rl import dcolt_loss, group_normalized_advantages, load_gpt2_tokenizer, official_sample_trace, rollout_record
from sedd_mini.official_prepare_data import build_sft_dataset as build_official_sft_dataset
from sedd_mini.official_prepare_data import save_dataset as save_official_dataset
from sedd_mini.sampling import sample_response, top_k_top_p_filter
from sedd_mini.tokenizer import ByteTokenizer
from sedd_mini.train import evaluate_loss as evaluate_mini_loss
from sedd_mini.utils import ExponentialMovingAverage, count_parameters, cycle, get_device, learning_rate, set_seed


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "sedd_mini").exists():
            return candidate
    raise FileNotFoundError(f"Could not find repo root from {start}")


PROJECT = find_project_root(Path.cwd())
os.chdir(PROJECT)

# Remote WSL proxy for Hugging Face downloads. Safe to leave set if unused.
os.environ.setdefault("PROXY", "http://172.27.0.1:7890")
os.environ.setdefault("HTTP_PROXY", os.environ["PROXY"])
os.environ.setdefault("HTTPS_PROXY", os.environ["PROXY"])

DEVICE_NAME = os.environ.get("DEVICE", "cuda" if torch.cuda.is_available() else "cpu")
DEVICE = get_device(DEVICE_NAME)
OFFICIAL_REPO = Path("external/Score-Entropy-Discrete-Diffusion")

SEQ_LEN = int(os.environ.get("SEQ_LEN", "256"))
ARC_SFT_LIMIT = int(os.environ.get("ARC_SFT_LIMIT", "0")) or None
ARC_RL_LIMIT = int(os.environ.get("ARC_RL_LIMIT", "0")) or None
MINI_PRETRAIN_STEPS = int(os.environ.get("MINI_PRETRAIN_STEPS", "20000"))
MINI_SFT_STEPS = int(os.environ.get("MINI_SFT_STEPS", "50"))
OFFICIAL_SFT_STEPS = int(os.environ.get("OFFICIAL_SFT_STEPS", "1000"))
LORA_RANK = int(os.environ.get("LORA_RANK", "8"))
LORA_ALPHA = float(os.environ.get("LORA_ALPHA", "16"))
LORA_DROPOUT = float(os.environ.get("LORA_DROPOUT", "0.05"))
RL_UPDATES = int(os.environ.get("RL_UPDATES", "100"))
DCOLT_NUM_GENERATIONS = int(os.environ.get("DCOLT_NUM_GENERATIONS", "4"))
DCOLT_REPEAT_TIMES = int(os.environ.get("DCOLT_REPEAT_TIMES", "1"))
DCOLT_CLIP_EPS = float(os.environ.get("DCOLT_CLIP_EPS", "0.2"))
DCOLT_BETA = float(os.environ.get("DCOLT_BETA", "0.02"))
SAMPLE_STEPS = int(os.environ.get("SAMPLE_STEPS", "4"))
MAX_NEW_TOKENS = int(os.environ.get("MAX_NEW_TOKENS", "12"))
TINYSTORIES_TRAIN_TEXTS = int(os.environ.get("TINYSTORIES_TRAIN_TEXTS", "8192"))
TINYSTORIES_VALID_TEXTS = int(os.environ.get("TINYSTORIES_VALID_TEXTS", "1024"))

DATA = PROJECT / "data" / "processed"
RUNS = PROJECT / "runs" / "arc_models"
for path in [DATA, RUNS / "base", RUNS / "arc_lora_sft", RUNS / "arc_dcolt_rl", RUNS / "evals"]:
    path.mkdir(parents=True, exist_ok=True)

print("project:", PROJECT)
print("device:", DEVICE)
print("cuda_available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
print("official_repo_exists:", OFFICIAL_REPO.exists())
print({
    "seq_len": SEQ_LEN,
    "arc_sft_limit": ARC_SFT_LIMIT,
    "arc_rl_limit": ARC_RL_LIMIT,
    "tinystories_train_texts": TINYSTORIES_TRAIN_TEXTS,
    "tinystories_valid_texts": TINYSTORIES_VALID_TEXTS,
    "mini_pretrain_steps": MINI_PRETRAIN_STEPS,
    "mini_sft_steps": MINI_SFT_STEPS,
    "official_sft_steps": OFFICIAL_SFT_STEPS,
    "lora_rank": LORA_RANK,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "rl_updates": RL_UPDATES,
    "sample_steps": SAMPLE_STEPS,
    "max_new_tokens": MAX_NEW_TOKENS,
    "dcolt_num_generations": DCOLT_NUM_GENERATIONS,
    "dcolt_repeat_times": DCOLT_REPEAT_TIMES,
    "dcolt_clip_eps": DCOLT_CLIP_EPS,
    "dcolt_beta": DCOLT_BETA,
})

## 1. Data Preparation

The compact model uses byte-tokenized TinyStories text for from-scratch pretraining. TinyStories is streamed from Hugging Face and capped by `TINYSTORIES_TRAIN_TEXTS` / `TINYSTORIES_VALID_TEXTS` so the cell remains lightweight. Official SEDD-small uses GPT-2-tokenized ARC-Easy records for SFT. ARC-Challenge records are saved as prompt/answer JSONL for exact-reward RL.

In [ ]:
set_seed(13)
mini_tokenizer = ByteTokenizer()


def load_tinystories_texts(split: str, limit: int) -> list[str]:
    from datasets import load_dataset

    rows = load_dataset("roneneldan/TinyStories", split=split, streaming=True)
    texts: list[str] = []
    for row in rows:
        text = str(row.get("text") or "").strip()
        if text:
            texts.append(text)
        if len(texts) >= limit:
            break
    if not texts:
        raise RuntimeError(f"No TinyStories texts loaded for split={split!r}")
    return texts


tinystories_train = load_tinystories_texts("train", TINYSTORIES_TRAIN_TEXTS)
tinystories_valid = load_tinystories_texts("validation", TINYSTORIES_VALID_TEXTS)

mini_pretrain_train = build_mini_pretrain_dataset(
    tinystories_train,
    mini_tokenizer,
    seq_len=128,
    shuffle=True,
)
mini_pretrain_valid = build_mini_pretrain_dataset(
    tinystories_valid,
    mini_tokenizer,
    seq_len=128,
    shuffle=False,
)
save_mini_dataset(mini_pretrain_train, DATA / "tinystories_pretrain_train.pt")
save_mini_dataset(mini_pretrain_valid, DATA / "tinystories_pretrain_valid.pt")

# The compact SFT stage remains a tiny instruction adaptation on top of the real pretraining checkpoint.
mini_sft = build_mini_sft_dataset(TOY_SFT_RECORDS, mini_tokenizer, seq_len=128)
mini_sft_train, mini_sft_valid = split_mini_dataset(mini_sft, valid_ratio=0.34, seed=13)
save_mini_dataset(mini_sft_train, DATA / "sft_train.pt")
save_mini_dataset(mini_sft_valid, DATA / "sft_valid.pt")

print("TinyStories texts", len(tinystories_train), len(tinystories_valid))
print("mini TinyStories pretrain", mini_pretrain_train.input_ids.shape, mini_pretrain_valid.input_ids.shape)
print("mini SFT", mini_sft_train.input_ids.shape, mini_sft_valid.input_ids.shape)
print("first TinyStories sample")
print(tinystories_train[0][:800])

In [ ]:
arc_easy_train = load_arc_records(config="ARC-Easy", split="train", limit=ARC_SFT_LIMIT)
arc_easy_valid = load_arc_records(config="ARC-Easy", split="validation", limit=None)
arc_challenge_train = load_arc_records(config="ARC-Challenge", split="train", limit=ARC_RL_LIMIT)
arc_challenge_valid = load_arc_records(config="ARC-Challenge", split="validation", limit=None)

arc_sft_train = build_official_sft_dataset(as_sft_records(arc_easy_train), seq_len=SEQ_LEN)
arc_sft_valid = build_official_sft_dataset(as_sft_records(arc_easy_valid), seq_len=SEQ_LEN)
save_official_dataset(arc_sft_train, DATA / "official_arc_easy_train.pt")
save_official_dataset(arc_sft_valid, DATA / "official_arc_easy_valid.pt")

save_mcqa_records(arc_challenge_train, DATA / "arc_challenge_rl_train.jsonl")
save_mcqa_records(arc_challenge_valid, DATA / "arc_challenge_rl_valid.jsonl")

print("ARC-Easy train/valid records", len(arc_easy_train), len(arc_easy_valid))
print("ARC-Challenge train/valid records", len(arc_challenge_train), len(arc_challenge_valid))
print("official SFT tensors", arc_sft_train.input_ids.shape, arc_sft_valid.input_ids.shape)
print("example prompt")
print(arc_easy_train[0].prompt)
print("target", arc_easy_train[0].response)

## 2. Compact From-scratch Training

This training loop is in the notebook so the mechanics are visible: build model, corrupt tokens with the absorbing mask process, apply the original score-entropy objective, update EMA, evaluate, and save checkpoints. The pretraining data is TinyStories, not toy strings.

In [ ]:
def train_mini_stage(
    *,
    config_path: str,
    out_dir: str,
    steps: int,
    batch_size: int,
    resume: str = "",
    train_path: str | None = None,
    valid_path: str | None = None,
) -> tuple[Path, list[dict[str, Any]]]:
    cfg = load_config(config_path)
    train_cfg = cfg["train"]
    train_cfg["out_dir"] = out_dir
    train_cfg["steps"] = steps
    train_cfg["batch_size"] = batch_size
    train_cfg["eval_every"] = steps
    train_cfg["save_every"] = steps
    train_cfg["log_every"] = max(1, min(10, steps))
    train_cfg["device"] = str(DEVICE)
    if train_path is not None:
        train_cfg["train_path"] = train_path
    if valid_path is not None:
        train_cfg["valid_path"] = valid_path
    if resume:
        train_cfg["resume"] = resume

    set_seed(int(train_cfg["seed"]))
    out_path = Path(out_dir)
    out_path.mkdir(parents=True, exist_ok=True)
    save_config(cfg, out_path / "config.yaml")

    train_data = TokenDataset(train_cfg["train_path"])
    valid_data = TokenDataset(train_cfg["valid_path"])
    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, collate_fn=collate_batch)
    valid_loader = DataLoader(valid_data, batch_size=batch_size, shuffle=False, collate_fn=collate_batch)

    model = build_model(cfg["model"]).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=float(train_cfg["lr"]), weight_decay=float(train_cfg["weight_decay"]))
    ema = ExponentialMovingAverage(model.parameters(), decay=float(train_cfg["ema_decay"]))
    start_step = 0

    if resume:
        payload = torch.load(resume, map_location=DEVICE)
        model.load_state_dict(payload["model"])
        if payload.get("config", {}).get("train", {}).get("stage") == train_cfg["stage"]:
            if "optimizer" in payload:
                optimizer.load_state_dict(payload["optimizer"])
            if "ema" in payload:
                ema.load_state_dict(payload["ema"])
            start_step = int(payload.get("step", 0))

    iterator = cycle(train_loader)
    use_amp = bool(train_cfg["amp"]) and DEVICE.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    logs: list[dict[str, Any]] = []
    model.train()
    t0 = time.time()

    print({"stage": train_cfg["stage"], "params": count_parameters(model), "train_rows": len(train_data), "valid_rows": len(valid_data)})
    for step in tqdm(range(start_step + 1, steps + 1), desc=f"mini {train_cfg['stage']}"):
        batch = next(iterator)
        input_ids = batch["input_ids"].to(DEVICE)
        loss_mask = batch["loss_mask"].to(DEVICE)
        lr = learning_rate(step, float(train_cfg["lr"]), int(train_cfg["warmup_steps"]))
        for group in optimizer.param_groups:
            group["lr"] = lr
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.bfloat16, enabled=use_amp):
            loss, metrics = score_entropy_loss(
                model,
                input_ids,
                loss_mask,
                mask_id=int(cfg["model"]["mask_id"]),
                eps=float(train_cfg["sampling_eps"]),
            )
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), float(train_cfg["grad_clip"]))
        scaler.step(optimizer)
        scaler.update()
        ema.update(model.parameters())

        if step % int(train_cfg["log_every"]) == 0 or step == steps:
            row = {"step": step, "loss": float(loss.detach().cpu()), "lr": lr, "elapsed_s": round(time.time() - t0, 2)}
            row.update(metrics)
            logs.append(row)
            print(row)

    ema.store(model.parameters())
    ema.copy_to(model.parameters())
    valid_loss = evaluate_mini_loss(
        model,
        valid_loader,
        device=DEVICE,
        mask_id=int(cfg["model"]["mask_id"]),
        eps=float(train_cfg["sampling_eps"]),
        max_batches=20,
    )
    ema.restore(model.parameters())
    print("valid_loss", valid_loss)

    checkpoint = out_path / "checkpoint_last.pt"
    save_checkpoint(checkpoint, model=model, config=cfg, step=steps, optimizer=optimizer, ema=ema)
    return checkpoint, logs

In [ ]:
mini_pretrain_ckpt, mini_pretrain_logs = train_mini_stage(
    config_path="configs/tiny_pretrain.yaml",
    out_dir="runs/notebook_tinystories_pretrain",
    steps=MINI_PRETRAIN_STEPS,
    batch_size=8,
    train_path=str(DATA / "tinystories_pretrain_train.pt"),
    valid_path=str(DATA / "tinystories_pretrain_valid.pt"),
)

mini_pretrain_model, mini_pretrain_cfg, _ = load_checkpoint(mini_pretrain_ckpt, device=DEVICE, use_ema=True)
print("TinyStories checkpoint", mini_pretrain_ckpt)

mini_sft_ckpt, mini_sft_logs = train_mini_stage(
    config_path="configs/tiny_sft.yaml",
    out_dir="runs/notebook_sft",
    steps=MINI_SFT_STEPS,
    batch_size=2,
    resume=str(mini_pretrain_ckpt),
)

mini_model, mini_cfg, _ = load_checkpoint(mini_sft_ckpt, device=DEVICE, use_ema=True)
mini_text = sample_response(
    mini_model,
    mini_tokenizer,
    "Explain SEDD briefly.",
    max_new_tokens=32,
    steps=4,
    temperature=0.9,
    top_k=50,
    top_p=0.95,
    seq_len=int(mini_cfg["model"]["seq_len"]),
    device=DEVICE,
)
print(mini_text)

## 3. TinyStories Zero-shot And Few-shot Samples

These examples use the compact model after TinyStories pretraining, before the tiny SEDD instruction SFT. They are story-continuation prompts, which match the real pretraining data distribution. The helper below continues the raw story prefix directly rather than wrapping it in a chat template. This compact byte-level model is mainly a from-scratch training mechanics demo; the official ARC model later in the notebook is the stronger capability demo.

In [ ]:
def mini_story_generate(prompt: str, *, model=mini_pretrain_model, max_new_tokens: int = 96) -> str:
    # Direct continuation for TinyStories. Do not use sample_response here because that helper
    # wraps prompts as User/Assistant, which is out-of-distribution for story pretraining.
    ids = torch.full(
        (1, int(mini_pretrain_cfg["model"]["seq_len"])),
        mini_tokenizer.pad_id,
        dtype=torch.long,
        device=DEVICE,
    )
    prompt_ids = mini_tokenizer.encode(prompt, add_bos=True, add_eos=False)
    max_prompt = max(1, ids.shape[1] - max_new_tokens)
    prompt_ids = prompt_ids[-max_prompt:]
    gen_len = min(max_new_tokens, ids.shape[1] - len(prompt_ids))
    ids[0, : len(prompt_ids)] = torch.tensor(prompt_ids, dtype=torch.long, device=DEVICE)
    response_slice = slice(len(prompt_ids), len(prompt_ids) + gen_len)
    ids[0, response_slice] = mini_tokenizer.mask_id
    response_mask = torch.zeros(ids.shape[1], dtype=torch.bool, device=DEVICE)
    response_mask[response_slice] = True

    model.eval()
    with torch.no_grad():
        for step in range(32):
            masked = (ids[0] == mini_tokenizer.mask_id) & response_mask
            positions = masked.nonzero(as_tuple=False).flatten()
            if positions.numel() == 0:
                break
            remaining_steps = max(1, 32 - step)
            count = max(1, int(torch.ceil(torch.tensor(positions.numel() / remaining_steps)).item()))
            fill_positions = positions[:count]
            t = torch.full((1,), 1.0 - (step / 32) * 0.999, device=DEVICE)
            sigma = loglinear_noise(t)[0]
            logits = model(ids.clone(), sigma)[0, fill_positions, : mini_tokenizer.mask_id]
            logits = top_k_top_p_filter(logits / 0.65, top_k=40, top_p=0.9)
            probs = torch.softmax(logits, dim=-1)
            sampled = torch.multinomial(probs, num_samples=1).squeeze(-1)
            ids[0, fill_positions] = sampled
    ids[ids == mini_tokenizer.mask_id] = mini_tokenizer.eos_id
    return mini_tokenizer.decode(ids[0, response_slice].tolist())


zero_shot_prompts = [
    "Once upon a time, a little fox found a shiny blue stone in the forest.",
    "Mia wanted to build the tallest tower, but her blocks kept falling down.",
]

few_shot_prompt = """Story: Tom had a red ball. It rolled under the bed. Tom asked his dad for help, and they found it together. Tom said thank you.

Story: Lily saw a bird with a hurt wing. She put it in a soft box and called her mom. The bird rested, then flew away.

Story: Ben opened the garden gate and saw a tiny door in the tree."""

print("ZERO-SHOT SAMPLES")
for prompt in zero_shot_prompts:
    print("\nPROMPT:", prompt)
    print("COMPLETION:", mini_story_generate(prompt))

print("\nFEW-SHOT SAMPLE")
print("PROMPT:", few_shot_prompt)
print("COMPLETION:", mini_story_generate(few_shot_prompt, max_new_tokens=112))

## 4. Official SEDD-small Base Checkpoint

The official checkpoint is loaded through the upstream SEDD code and exported to a local `.pt` artifact so the base, SFT, and RL checkpoints have the same evaluation interface.

In [ ]:
if not OFFICIAL_REPO.exists():
    raise FileNotFoundError(
        f"Official repo not found at {OFFICIAL_REPO}. Prepare it once before running official-model cells."
    )

base_model, base_graph, base_noise, base_model_path, base_step = load_official_components(
    "louaaron/sedd-small",
    repo_path=OFFICIAL_REPO,
    device=DEVICE,
)
base_ckpt = RUNS / "base" / "checkpoint_base.pt"
save_official_checkpoint(
    base_ckpt,
    model=base_model,
    base_model_path=base_model_path,
    step=base_step,
    extra={"stage": "base_export"},
)
print("saved", base_ckpt)
del base_model, base_graph, base_noise
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

## 5. Official ARC-Easy LoRA SFT Loop

The loss is response-only score entropy: prompt and answer choices remain visible/clean, and only the target `Answer: <letter>` tokens contribute to the denoising score objective. The trainable parameters are LoRA adapters on the SEDD attention and MLP linear layers; the saved checkpoint is merged back into a normal SEDD state dict for serving.

In [ ]:
def official_sft_python(
    *,
    model_path: str,
    train_path: str,
    valid_path: str,
    out_dir: str,
    steps: int,
    batch_size: int = 1,
    lr: float = 1.0e-5,
    eval_every: int = 50,
    max_eval_batches: int = 50,
    lora_rank: int = LORA_RANK,
    lora_alpha: float = LORA_ALPHA,
    lora_dropout: float = LORA_DROPOUT,
) -> tuple[Path, list[dict[str, Any]]]:
    model, graph, noise, base_model_path, loaded_step = load_official_components(
        model_path,
        repo_path=OFFICIAL_REPO,
        device=DEVICE,
    )
    lora_targets = ["attn_qkv", "attn_out", "mlp.0", "mlp.2"]
    lora_modules = apply_lora(
        model,
        rank=lora_rank,
        alpha=lora_alpha,
        dropout=lora_dropout,
        targets=lora_targets,
    )
    model.train()
    train_data = TokenDataset(train_path)
    valid_data = TokenDataset(valid_path)
    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, collate_fn=collate_batch)
    valid_loader = DataLoader(valid_data, batch_size=batch_size, shuffle=False, collate_fn=collate_batch)
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=0.01)
    iterator = cycle(train_loader)
    logs: list[dict[str, Any]] = []

    print({
        "loaded_step": loaded_step,
        "train_rows": len(train_data),
        "valid_rows": len(valid_data),
        "lora_modules": len(lora_modules),
        "trainable_parameters": trainable_parameter_count(model),
        "total_parameters": total_parameter_count(model),
    })
    for step in tqdm(range(1, steps + 1), desc="official arc lora sft"):
        batch = next(iterator)
        clean = batch["input_ids"].to(DEVICE)
        loss_mask = batch["loss_mask"].to(DEVICE)
        for group in optimizer.param_groups:
            group["lr"] = learning_rate(step, lr, warmup_steps=50)
        optimizer.zero_grad(set_to_none=True)
        loss, metrics = official_score_entropy_loss(model, graph, noise, clean, loss_mask)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        if step % 25 == 0 or step == steps:
            row = {"step": step, "loss": float(loss.detach().cpu()), "lr": optimizer.param_groups[0]["lr"]}
            row.update(metrics)
            logs.append(row)
            print(row)
        if step % eval_every == 0 or step == steps:
            valid_loss = evaluate_official_loss(
                model,
                graph,
                noise,
                valid_loader,
                device=DEVICE,
                max_batches=max_eval_batches,
            )
            print({"step": step, "valid_score_entropy": valid_loss})

    out_path = Path(out_dir)
    out_path.mkdir(parents=True, exist_ok=True)
    checkpoint = out_path / "checkpoint_last.pt"
    save_merged_lora_checkpoint(
        checkpoint,
        model=model,
        base_model_path=base_model_path,
        step=steps,
        optimizer=optimizer,
        extra={
            "source_model_path": model_path,
            "stage": "arc_lora_sft",
            "lora_rank": lora_rank,
            "lora_alpha": lora_alpha,
            "lora_dropout": lora_dropout,
            "lora_targets": lora_targets,
        },
    )
    del model, graph, noise
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return checkpoint, logs

In [ ]:
sft_ckpt, sft_logs = official_sft_python(
    model_path=str(base_ckpt),
    train_path=str(DATA / "official_arc_easy_train.pt"),
    valid_path=str(DATA / "official_arc_easy_valid.pt"),
    out_dir=str(RUNS / "arc_lora_sft"),
    steps=OFFICIAL_SFT_STEPS,
    batch_size=1,
    lora_rank=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
)
print("saved", sft_ckpt)

## 6. ARC-Challenge DCoLT RL Loop

Following the DCoLT/LLaDOU training shape, each prompt is sampled multiple times through the reverse diffusion chain. The group of completed answers receives exact answer-key rewards, rewards are normalized within the group, and the SEDD denoising trajectory log probabilities are optimized with a clipped outcome-RL objective plus reference KL.

In [ ]:
def official_arc_dcolt_rl_python(
    *,
    model_path: str,
    reference_model_path: str,
    records_path: str,
    out_dir: str,
    updates: int,
    seq_len: int,
    max_new_tokens: int,
    sample_steps: int,
    batch_size: int = 1,
    num_generations: int = DCOLT_NUM_GENERATIONS,
    repeat_times: int = DCOLT_REPEAT_TIMES,
    lr: float = 5.0e-6,
    clip_eps: float = DCOLT_CLIP_EPS,
    beta: float = DCOLT_BETA,
) -> tuple[Path, list[dict[str, Any]]]:
    try:
        from transformers import GPT2TokenizerFast
    except Exception as exc:
        raise RuntimeError("Official RL requires transformers.") from exc

    model, graph, noise, base_model_path, loaded_step = load_official_components(
        model_path,
        repo_path=OFFICIAL_REPO,
        device=DEVICE,
    )
    reference_model, _, _, _, _ = load_official_components(
        reference_model_path,
        repo_path=OFFICIAL_REPO,
        device=DEVICE,
    )
    for param in reference_model.parameters():
        param.requires_grad_(False)
    model.train()
    reference_model.eval()

    tokenizer = load_gpt2_tokenizer()
    records = load_mcqa_records(records_path)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    mask_id = int(graph.dim - 1)
    seq_len = min(seq_len, int(model.config.model.length))
    rollouts_per_prompt = max(1, num_generations) * max(1, repeat_times)
    logs: list[dict[str, Any]] = []

    print({
        "loaded_step": loaded_step,
        "records": len(records),
        "seq_len": seq_len,
        "rollouts_per_prompt": rollouts_per_prompt,
        "clip_eps": clip_eps,
        "beta": beta,
    })
    for update in tqdm(range(1, updates + 1), desc="official arc dcolt"):
        optimizer.zero_grad(set_to_none=True)
        update_rollouts = []
        update_advantages: list[float] = []

        for _ in range(batch_size):
            record = random.choice(records)
            group = [
                rollout_record(
                    model=model,
                    reference_model=reference_model,
                    noise=noise,
                    tokenizer=tokenizer,
                    record=record,
                    seq_len=seq_len,
                    mask_id=mask_id,
                    max_new_tokens=max_new_tokens,
                    steps=sample_steps,
                    temperature=1.0,
                    top_k=50,
                    top_p=0.95,
                    device=DEVICE,
                )
                for _ in range(rollouts_per_prompt)
            ]
            update_rollouts.extend(group)
            update_advantages.extend(group_normalized_advantages([rollout.reward for rollout in group]))

        rewards = [rollout.reward for rollout in update_rollouts]
        valid_samples = sum(1 for advantage in update_advantages if abs(advantage) > 1.0e-8)
        if valid_samples == 0:
            row = {
                "update": update,
                "skipped": True,
                "reason": "zero_group_variance",
                "reward": sum(rewards) / max(len(rewards), 1),
            }
            logs.append(row)
            print(row)
            continue

        loss, metrics = dcolt_loss(
            update_rollouts,
            update_advantages,
            clip_eps=clip_eps,
            beta=beta,
            entropy_coef=0.0,
        )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        if update % 5 == 0 or update == updates:
            sample = update_rollouts[0]
            row = {
                "update": update,
                "loss": float(loss.detach().cpu()),
                "reward": sum(rewards) / max(len(rewards), 1),
                "reward_min": min(rewards),
                "reward_max": max(rewards),
                "valid_samples": valid_samples,
                "gold": sample.gold,
                "pred": sample.prediction or "?",
                "sample": sample.text[:120],
            }
            row.update(metrics)
            logs.append(row)
            print(row)

    out_path = Path(out_dir)
    out_path.mkdir(parents=True, exist_ok=True)
    checkpoint = out_path / "checkpoint_last.pt"
    save_official_checkpoint(
        checkpoint,
        model=model,
        base_model_path=base_model_path,
        step=updates,
        optimizer=optimizer,
        extra={
            "source_model_path": model_path,
            "reference_model_path": reference_model_path,
            "stage": "dcolt_rl",
            "num_generations": num_generations,
            "repeat_times": repeat_times,
            "clip_eps": clip_eps,
            "beta": beta,
        },
    )
    del model, reference_model, graph, noise
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return checkpoint, logs

In [ ]:
rl_ckpt, rl_logs = official_arc_dcolt_rl_python(
    model_path=str(sft_ckpt),
    reference_model_path=str(base_ckpt),
    records_path=str(DATA / "arc_challenge_rl_train.jsonl"),
    out_dir=str(RUNS / "arc_dcolt_rl"),
    updates=RL_UPDATES,
    seq_len=SEQ_LEN,
    max_new_tokens=MAX_NEW_TOKENS,
    sample_steps=SAMPLE_STEPS,
    batch_size=1,
    num_generations=DCOLT_NUM_GENERATIONS,
    repeat_times=DCOLT_REPEAT_TIMES,
)
print("saved", rl_ckpt)

## 7. Evaluation

Score entropy evaluates fit to held-out ARC-Easy SFT targets. Exact reward evaluates generated answer letters on ARC validation records.

In [ ]:
def evaluate_official_mcqa_python(
    *,
    model_path: str,
    data_path: str,
    records: list[MCQARecord],
    reward_samples: int = 32,
    seq_len: int = SEQ_LEN,
    max_new_tokens: int = MAX_NEW_TOKENS,
    sample_steps: int = SAMPLE_STEPS,
) -> dict[str, Any]:
    try:
        from transformers import GPT2TokenizerFast
    except Exception as exc:
        raise RuntimeError("MCQA evaluation requires transformers.") from exc

    model, graph, noise, base_model_path, step = load_official_components(
        model_path,
        repo_path=OFFICIAL_REPO,
        device=DEVICE,
    )
    dataset = TokenDataset(data_path)
    loader = DataLoader(dataset, batch_size=1, shuffle=False, collate_fn=collate_batch)
    score_entropy = evaluate_official_loss(model, graph, noise, loader, device=DEVICE, max_batches=50)

    tokenizer = load_gpt2_tokenizer()
    mask_id = int(graph.dim - 1)
    seq_len = min(seq_len, int(model.config.model.length))
    rewards: list[float] = []
    outputs: list[dict[str, str]] = []
    with torch.no_grad():
        for record in records[:reward_samples]:
            trace = official_sample_trace(
                model,
                model,
                noise,
                tokenizer,
                record.prompt,
                seq_len=seq_len,
                mask_id=mask_id,
                max_new_tokens=max_new_tokens,
                steps=sample_steps,
                temperature=1.0,
                top_k=50,
                top_p=0.95,
                device=DEVICE,
            )
            text = tokenizer.decode(trace.response_ids.tolist(), skip_special_tokens=True)
            pred = extract_choice(text, record.labels)
            rewards.append(exact_choice_reward(text, record.answer, record.labels))
            outputs.append({"gold": record.answer, "pred": pred, "text": text[:160]})

    result = {
        "model_path": model_path,
        "base_model_path": base_model_path,
        "step": step,
        "score_entropy": score_entropy,
        "exact_reward": sum(rewards) / max(len(rewards), 1),
        "num_reward_samples": len(rewards),
        "sample_outputs": outputs[:5],
    }
    del model, graph, noise
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return result

In [ ]:
eval_results = {
    "base_arc_easy": evaluate_official_mcqa_python(
        model_path=str(base_ckpt),
        data_path=str(DATA / "official_arc_easy_valid.pt"),
        records=arc_easy_valid,
        reward_samples=32,
    ),
    "lora_sft_arc_easy": evaluate_official_mcqa_python(
        model_path=str(sft_ckpt),
        data_path=str(DATA / "official_arc_easy_valid.pt"),
        records=arc_easy_valid,
        reward_samples=32,
    ),
    "dcolt_rl_arc_challenge": evaluate_official_mcqa_python(
        model_path=str(rl_ckpt),
        data_path=str(DATA / "official_arc_easy_valid.pt"),
        records=arc_challenge_valid,
        reward_samples=32,
    ),
}
print(json.dumps(eval_results, indent=2, ensure_ascii=False))

## 8. Inference

The cell output is the demonstration surface. Each call uses the official diffusion backend; tokens are produced through reverse denoising steps rather than left-to-right autoregressive decoding.

In [ ]:
def generate_official(model_path: str, prompt: str) -> str:
    backend = OfficialSEDDBackend(
        model_path=model_path,
        repo_path=OFFICIAL_REPO,
        device_name=str(DEVICE),
    )
    text = backend.generate(
        prompt,
        GenerationParams(
            max_new_tokens=MAX_NEW_TOKENS,
            steps=max(SAMPLE_STEPS, 4),
            temperature=0.9,
            top_k=50,
            top_p=0.95,
        ),
    )
    del backend
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return text

example = arc_challenge_valid[0]
print(example.prompt)
print("gold", example.answer)
for name, model_path in {
    "base": str(base_ckpt),
    "lora_sft": str(sft_ckpt),
    "dcolt_rl": str(rl_ckpt),
}.items():
    print("\n" + "=" * 80)
    print(name)
    output = generate_official(model_path, example.prompt)
    print(output)
    print("parsed", extract_choice(output, example.labels))

## 9. Artifacts

The main outputs are separate checkpoint files for the base export, ARC LoRA SFT model, and ARC DCoLT RL model. These are enough for later evaluation or serving.

In [ ]:
artifacts = {
    "mini_tinystories_pretrain": str(mini_pretrain_ckpt),
    "mini_sft": str(mini_sft_ckpt),
    "tinystories_train_data": str(DATA / "tinystories_pretrain_train.pt"),
    "tinystories_valid_data": str(DATA / "tinystories_pretrain_valid.pt"),
    "official_base": str(base_ckpt),
    "official_arc_lora_sft": str(sft_ckpt),
    "official_arc_dcolt_rl": str(rl_ckpt),
    "arc_easy_train_data": str(DATA / "official_arc_easy_train.pt"),
    "arc_easy_valid_data": str(DATA / "official_arc_easy_valid.pt"),
    "arc_challenge_rl_train": str(DATA / "arc_challenge_rl_train.jsonl"),
    "arc_challenge_rl_valid": str(DATA / "arc_challenge_rl_valid.jsonl"),
}
print(json.dumps(artifacts, indent=2))